# Financial workstream: UK-listed companies data foundation

This is the shared starting point for the financial side of the project. It builds a small, clean dataset of UK-listed companies with their share-price behaviour and basic financial figures, which we then each analyse in our own notebook.

**Why listed companies, and an honest note.** The market data APIs (Yahoo Finance and the rest) only cover companies listed on a stock exchange. Our target customers are private small and medium firms, which have no share price at all. So we use the listed UK companies as a worked example and a place where the brief's two ideas can actually be tested: does media sentiment line up with share-price performance, and do financial ratios predict outcomes. The findings for private firms come from Companies House data, which is a separate piece.

**No API key needed.** This notebook uses `yfinance`, which is free and needs no key, so anyone on the team can run it straight away. Install once with `pip install yfinance`.

**How we split the work (so we do not edit the same file):**
- This notebook builds the shared dataset and saves it to `data/processed/listed_companies.csv`.
- Each person then makes their own notebook that loads that file:
  - one of us links the companies to news sentiment (GDELT) and checks it against share-price moves,
  - the other works on the financial ratios and a first predictive model.
- Because we work in separate notebooks that read the same saved file, we avoid clashing on the same cells.

## Step 1: The company list

A small, editable set of UK-listed firms tagged with our project sectors. Add or remove names freely; the method matters more than the exact list.

In [1]:
import pandas as pd
import yfinance as yf

# A small, editable list of UK-listed companies across our target sectors.
# Add or remove tickers freely (London tickers end in ".L"). This is the shared
# starting set; the point is the method, not the exact list.
TICKERS = {
    # Manufacturing
    "RR.L": "Manufacturing", "BA.L": "Manufacturing", "SMIN.L": "Manufacturing",
    "WEIR.L": "Manufacturing", "MRO.L": "Manufacturing",
    # Technology, legal & professional
    "SGE.L": "Technology, legal & professional", "AUTO.L": "Technology, legal & professional",
    "RMV.L": "Technology, legal & professional", "EXPN.L": "Technology, legal & professional",
    "REL.L": "Technology, legal & professional", "BNZL.L": "Technology, legal & professional",
    # Healthcare
    "SN.L": "Healthcare", "HIK.L": "Healthcare", "GSK.L": "Healthcare", "AZN.L": "Healthcare",
    # Wholesale & retail
    "TSCO.L": "Wholesale & retail", "SBRY.L": "Wholesale & retail", "NXT.L": "Wholesale & retail",
    "MKS.L": "Wholesale & retail", "GRG.L": "Wholesale & retail", "JD.L": "Wholesale & retail",
    # Real estate
    "LAND.L": "Real estate", "BLND.L": "Real estate", "SGRO.L": "Real estate",
    # Fast growth & emerging
    "OCDO.L": "Fast growth & emerging", "WISE.L": "Fast growth & emerging",
}
print(f"{len(TICKERS)} companies across {len(set(TICKERS.values()))} sectors")

26 companies across 6 sectors


## Step 2: Pull share prices and basic financials

For each company we pull, from Yahoo Finance: a few headline figures (market value, profitability, debt, growth) and, from one year of prices, the total return and how volatile the price was. Companies that fail to fetch are skipped, so a couple of gaps do not stop the run.

In [1]:
rows = []
for ticker, sector in TICKERS.items():
    try:
        t = yf.Ticker(ticker)
        info = t.info
        hist = t.history(period="1y")

        ret_1y = vol_1y = None
        if len(hist) > 20:
            closes = hist["Close"].dropna()
            ret_1y = round(float(closes.iloc[-1] / closes.iloc[0] - 1), 3)         # 1 year total return
            vol_1y = round(float(closes.pct_change().std() * (252 ** 0.5)), 3)      # annualised volatility

        rows.append({
            "ticker": ticker,
            "name": info.get("shortName"),
            "sector": sector,                       # our project sector label
            "market_cap": info.get("marketCap"),
            "trailing_pe": info.get("trailingPE"),
            "price_to_book": info.get("priceToBook"),
            "profit_margin": info.get("profitMargins"),
            "return_on_equity": info.get("returnOnEquity"),
            "debt_to_equity": info.get("debtToEquity"),
            "revenue_growth": info.get("revenueGrowth"),
            "return_1y": ret_1y,
            "volatility_1y": vol_1y,
        })
        print(f"  ok  {ticker:8s} {info.get('shortName')}")
    except Exception as e:
        print(f"  skip {ticker:8s} ({e})")

market = pd.DataFrame(rows)
print(f"\nBuilt dataset: {market.shape[0]} companies, {market.shape[1]} columns")

  ok  RR.L     ROLLS-ROYCE HOLDINGS PLC ORD SH
  ok  BA.L     BAE SYSTEMS PLC ORD 2.5P
  ok  SMIN.L   SMITHS GROUP PLC ORD 37.5P
  ok  WEIR.L   WEIR GROUP PLC ORD 12.5P
  ok  MRO.L    MELROSE INDUSTRIES PLC ORD GBP0
  ok  SGE.L    THE SAGE GROUP PLC ORD 1 4/77P
  ok  AUTO.L   AUTOTRADER GROUP PLC ORD 1P
  ok  RMV.L    RIGHTMOVE PLC ORD 0.1P
  ok  EXPN.L   EXPERIAN PLC ORD USD0.10
  ok  REL.L    RELX PLC ORD 14 51/116P
  ok  BNZL.L   BUNZL PLC ORD 32 1/7P
  ok  SN.L     SMITH & NEPHEW PLC ORD USD0.20
  ok  HIK.L    HIKMA PHARMACEUTICALS PLC ORD S
  ok  GSK.L    GSK PLC ORD 31 1/4P
  ok  AZN.L    ASTRAZENECA PLC ORD SHS $0.25
  ok  TSCO.L   TESCO PLC ORD 6 1/3P
  ok  SBRY.L   SAINSBURY (J) PLC ORD 28 4/7P
  ok  NXT.L    NEXT PLC ORD 10P
  ok  MKS.L    MARKS AND SPENCER GROUP PLC ORD
  ok  GRG.L    GREGGS PLC ORD 2P
  ok  JD.L     JD SPORTS FASHION PLC ORD 0.05P
  ok  LAND.L   LAND SECURITIES GROUP PLC ORD 1
  ok  BLND.L   BRITISH LAND COMPANY PLC ORD 25
  ok  SGRO.L   SEGRO PLC ORD 10P
 

In [1]:
# A quick look at the result
cols = ["ticker", "name", "sector", "market_cap", "profit_margin", "return_1y", "volatility_1y"]
print(market[cols].to_string(index=False))

ticker                            name                           sector   market_cap  profit_margin  return_1y  volatility_1y
  RR.L ROLLS-ROYCE HOLDINGS PLC ORD SH                    Manufacturing 116154245120        0.27543      0.587          0.367
  BA.L        BAE SYSTEMS PLC ORD 2.5P                    Manufacturing  54163804160        0.07277     -0.038          0.303
SMIN.L      SMITHS GROUP PLC ORD 37.5P                    Manufacturing   7733170688        0.08706      0.164          0.224
WEIR.L        WEIR GROUP PLC ORD 12.5P                    Manufacturing   6365687296        0.09628     -0.006          0.289
 MRO.L MELROSE INDUSTRIES PLC ORD GBP0                    Manufacturing   5953577984        0.10309     -0.043          0.341
 SGE.L  THE SAGE GROUP PLC ORD 1 4/77P Technology, legal & professional   7285234176        0.14617     -0.358          0.286
AUTO.L     AUTOTRADER GROUP PLC ORD 1P Technology, legal & professional   3796362496        0.47077     -0.409        

## Step 3: Save the shared dataset

We save the table so each of us can load it in our own analysis notebook.

In [1]:
from pathlib import Path
Path("../data/processed").mkdir(parents=True, exist_ok=True)
OUT = "../data/processed/listed_companies.csv"
market.to_csv(OUT, index=False)
print(f"Saved {len(market)} companies to {OUT}")
print("\nCompanies per sector:")
print(market["sector"].value_counts().to_string())

Saved 26 companies to ../data/processed/listed_companies.csv

Companies per sector:
sector
Technology, legal & professional    6
Wholesale & retail                  6
Manufacturing                       5
Healthcare                          4
Real estate                         3
Fast growth & emerging              2


## What comes next

With this shared dataset in place, the two pieces are:

1. **Market sentiment vs performance.** Link each company to its news coverage (using the GDELT method from the rest of the project), score the sentiment, and check whether more positive or negative coverage lines up with better or worse share-price returns. This is the brief's "correlate stock performance with media coverage".

2. **Financial health and a first model.** Use the ratios here (profitability, debt, growth) to look for patterns, and build a simple model to predict an outcome such as next-period return direction. This is the brief's "use ratios and trends for predictive modelling".

**Limitation to keep in view:** all of this is for listed firms. It does not reach the private SME target population, where the only financial data is the Companies House accounts. We will say this plainly in the write-up.